# Benchmark with interaction terms

[](library_comparison.ipynb) holds mboostrs to a sum of one-dimensional terms
while XGBoost, LightGBM, CatBoost, EBM and the MLP choose their own
interactions. That makes its result ambiguous: the gap could be the library or
it could be the restriction.

This study removes the restriction. Same six datasets, same three 75/25 splits
per dataset, same seeds, same twelve tuning configurations scored by the same
three-fold inner cross-validation. mboostrs is given every two-way interaction
as a candidate base learner and its selection step — the one already choosing
among main effects at each iteration — decides which to use.

Two mboostrs entries appear. `mboostrs (all pairs)` is offered every pair, which is called the exhaustive search below.
`mboostrs (screened)` is offered only the twenty pairs that rank highest on a
split-sample test of the additive model's residuals, which costs no model fits.
`mboostrs GAMBoost` is the additive fit, carried over for reference.

This page examines what changes when every library may fit interaction terms. The
interaction formulas are listed first so the compared model classes can be checked for similarity.


:::{admonition} What this study estimates
:class: dropdown

| | |
|---|---|
| **Target** | whether the previous comparison's gap is the library or the restriction to one-dimensional terms |
| **Observed response** | the target column of each dataset |
| **Estimand** | the change in held-out error when mboostrs is allowed interactions, holding everything else fixed |
| **Units** | the metric's own units per dataset |
| **Sampling unit** | one row of each dataset |
| **Dependence** | treated as independent within each dataset |
| **Split / resampling unit** | the row, matching the companion study |
| **Tuning** | the same budget as the companion study |
| **Uncertainty** | across-split variation |
| **Claim type** | predictive comparison |
:::


In [1]:
import json
import sys
from pathlib import Path
# Read the recorded experiment instead of rerunning its expensive fits.
import numpy as np
import polars as pl
import altair as alt

sys.path.insert(0, str(Path("../../../benches").resolve()))

libraries = pl.DataFrame(
    json.loads(Path("bench_libraries_clipped.json").read_text())["results"])
search = pl.DataFrame(
    json.loads(Path("bench_interaction_search_v2.json").read_text())["results"])

others = (libraries.filter(pl.col("library") != "XGBoost depth 1")
          .select("dataset", "repeat", "library", "score", "total_seconds"))
mine = pl.concat([
    search.filter(pl.col("variant") == v)
          .select("dataset", "repeat", pl.lit(name).alias("library"),
                  "score", "total_seconds")
    for v, name in (("all_pairs", "mboostrs (all pairs)"),
                    ("screened", "mboostrs (screened)"))])
runs = pl.concat([others, mine])

In [2]:
ORDER = ["mboostrs (all pairs)", "mboostrs (screened)", "mboostrs GAMBoost",
         "EBM", "XGBoost", "LightGBM", "CatBoost", "MLP"]
by_dataset = runs.group_by("dataset", "library").agg(pl.col("score").mean())
print(f"{runs.height} runs: 8 models x 6 datasets x 3 splits")
print(f"failures: {search.filter(pl.col('score').is_null()).height}, "
      f"fits whose training risk rose: {search.filter(pl.col('diverged')).height}")

144 runs: 8 models x 6 datasets x 3 splits
failures: 0, fits whose training risk rose: 0


## Summary of results

The table gives each model's held-out score on each data set, its mean
rank and its time to tune and refit; the two mboostrs interaction variants
are the rows this study adds to the companion comparison.

In [3]:
from bench_tables import table as annotated

SCORE_NOTE = ("Scores are the mean over three random 75/25 splits shared by every model: "
              "test R² for california, cpu_act and diabetes, test ROC AUC for breast_cancer, "
              "credit_g and electricity; both are higher-is-better with 1 the maximum.")
DATASETS = ["california", "cpu_act", "diabetes", "breast_cancer", "credit_g", "electricity"]
summary_rows = (by_dataset.pivot(on="dataset", index="library", values="score")
                .join(by_dataset.with_columns((-pl.col("score")).rank("average")
                                              .over("dataset").alias("rank"))
                      .group_by("library").agg(pl.col("rank").mean()), on="library")
                .join(runs.group_by("library").agg(pl.col("total_seconds").mean()
                                                   .alias("seconds")), on="library")
                .select("library", *DATASETS, "rank", "seconds").sort("rank"))

In [4]:
annotated(
    summary_rows, stub="library", stubhead="Library",
    title="Eight models on six data sets, with mboostrs interaction terms",
    subtitle="Held-out score, mean over 3 random 75/25 splits",
    spanners={"R² ↑": DATASETS[:3], "AUC ↑": DATASETS[3:]},
    labels={"rank": "mean rank ↓", "seconds": "s ↓"},
    better={**{d: "max" for d in DATASETS}, "rank": "min", "seconds": "min"},
    formats={"rank": 2, "seconds": 1},
    notes=[SCORE_NOTE, "mboostrs (all pairs): every pair of predictors offered as a "
           "boosting term; (screened): only pairs a residual screen proposes; GAMBoost: "
           "main effects only. s: mean seconds to tune and refit (the mboostrs variants "
           "come from a separate run). Bold: best in the column. Sources: "
           "bench_libraries_clipped.json and bench_interaction_search_v2.json."])

GT(_tbl_data=shape: (8, 9)
┌────────────┬────────────┬──────────┬──────────┬───┬──────────┬────────────┬──────────┬───────────┐
│ library    ┆ california ┆ cpu_act  ┆ diabetes ┆ … ┆ credit_g ┆ electricit ┆ rank     ┆ seconds   │
│ ---        ┆ ---        ┆ ---      ┆ ---      ┆   ┆ ---      ┆ y          ┆ ---      ┆ ---       │
│ str        ┆ f64        ┆ f64      ┆ f64      ┆   ┆ f64      ┆ ---        ┆ f64      ┆ f64       │
│            ┆            ┆          ┆          ┆   ┆          ┆ f64        ┆          ┆           │
╞════════════╪════════════╪══════════╪══════════╪═══╪══════════╪════════════╪══════════╪═══════════╡
│ CatBoost   ┆ 0.827827   ┆ 0.984909 ┆ 0.447118 ┆ … ┆ 0.793194 ┆ 0.918455   ┆ 2.666667 ┆ 14.863889 │
│ EBM        ┆ 0.80842    ┆ 0.984704 ┆ 0.484613 ┆ … ┆ 0.795479 ┆ 0.908458   ┆ 3.0      ┆ 106.91388 │
│            ┆            ┆          ┆          ┆   ┆          ┆            ┆          ┆ 9         │
│ XGBoost    ┆ 0.813615   ┆ 0.985283 ┆ 0.411659 ┆ … ┆ 0.788978 ┆ 0.917252   ┆ 4.0      ┆ 18.665556 │
│ LightGBM   ┆ 0.814093   ┆ 0.982822 ┆ 0.417331 ┆ … ┆ 0.786794 ┆ 0.918378   ┆ 4.0      ┆ 36.805556 │
│ mboostrs   ┆ 0.75872    ┆ 0.985061 ┆ 0.486963 ┆ … ┆ 0.771505 ┆ 0.873683   ┆ 4.333333 ┆ 380.59222 │
│ (all       ┆            ┆          ┆          ┆   ┆          ┆            ┆          ┆ 2         │
│ pairs)     ┆            ┆          ┆          ┆   ┆          ┆            ┆          ┆           │
│ mboostrs   ┆ 0.718742   ┆ 0.980789 ┆ 0.499496 ┆ … ┆ 0.789257 ┆ 0.85954    ┆ 5.333333 ┆ 0.947222  │
│ GAMBoost   ┆            ┆          ┆          ┆   ┆          ┆            ┆          ┆           │
│ mboostrs   ┆ 0.749461   ┆ 0.983452 ┆ 0.48859  ┆ … ┆ 0.780978 ┆ 0.867203   ┆ 5.5      ┆ 18.477778 │
│ (screened) ┆            ┆          ┆          ┆   ┆          ┆            ┆          ┆           │
│ MLP        ┆ 0.732678   ┆ 0.97563  ┆ 0.452435 ┆ … ┆ 0.767213 ┆ 0.853349   ┆ 7.166667 ┆ 6.215     │
└────────────┴────────────┴──────────┴──────────┴───┴──────────┴────────────┴──────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x79faea5b5070>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None, stub_level=0), ColInfo(var='california', type=<ColInfoTypeEnum.default: 1>, column_label='california', column_align='right', column_width=None, stub_level=None), ColInfo(var='cpu_act', type=<ColInfoTypeEnum.default: 1>, column_label='cpu_act', column_align='right', column_width=None, stub_level=None), ColInfo(var='diabetes', type=<ColInfoTypeEnum.default: 1>, column_label='diabetes', column_align='right', column_width=None, stub_level=None), ColInfo(var='breast_cancer', type=<ColInfoTypeEnum.default: 1>, column_label='breast_cancer', column_align='right', column_width=None, stub_level=None), ColInfo(var='credit_g', type=<ColInfoTypeEnum.default: 1>, column_label='credit_g', column_align='right', column_width=None, stub_level=None), ColInfo(var='electricity', type=<ColInfoTypeEnum.default: 1>, column_label='electricity', column_align='right', column_width=None, stub_level=None), ColInfo(var='rank', type=<ColInfoTypeEnum.default: 1>, column_label='mean rank ↓', column_align='right', column_width=None, stub_level=None), ColInfo(var='seconds', type=<ColInfoTypeEnum.default: 1>, column_label='s ↓', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79fb31c9f200>, _spanners=Spanners([SpannerInfo(spanner_id='R² ↑', spanner_level=0, spanner_label='R² ↑', spanner_units=None, spanner_pattern=None, vars=['california', 'cpu_act', 'diabetes'], built=None), SpannerInfo(spanner_id='AUC ↑', spanner_level=0, spanner_label='AUC ↑', spanner_units=None, spanner_pattern=None, vars=['breast_cancer', 'credit_g', 'electricity'], built=None)]), _heading=Heading(title='Eight models on six data sets, with mboostrs interaction terms', subtitle='Held-out score, mean over 3 random 75/25 splits', pre

## Interaction formulas

The interaction variants are one `gamboost` call with one formula. For
california that formula holds 8 main effects and 28 interactions; the terms are
built by the same functions the benchmark used, imported here rather than
retyped.

The learner for a pair follows from its two margins. Two continuous margins get
a tensor product (a surface over both columns, written `a %X% b` in the formula); a continuous margin paired with an indicator gets a
varying coefficient `bbs(a, by=b)`; two indicators get a product
`bols(a, by=b)`. A two-valued predictor needs only the contrast between its levels; the linear
and varying-coefficient terms represent that contrast without a redundant
spline basis.

In [5]:
from bench_interaction_search import levels_of, main_terms, pair_term
import bench_libraries as bench
from sklearn.datasets import fetch_california_housing

names = fetch_california_housing().feature_names
_, X_cal, _ = bench.encoded_datasets()["california"]
levels = levels_of(X_cal)
p = X_cal.shape[1]
pairs = [(a, b) for a in range(p) for b in range(a + 1, p)]
terms = main_terms(levels) + [pair_term(a, b, levels, 4.0, 10) for a, b in pairs]

spec = "y ~ " + "\n    + ".join(terms)
for j, nm in enumerate(names):
    spec = spec.replace(f"f{j},", f"{nm},").replace(f"f{j})", f"{nm})")
lines = spec.splitlines()
print(f"california: {p} predictors -> {len(main_terms(levels))} main effects "
      f"+ {len(pairs)} interactions = {len(terms)} base learners\n")
print("\n".join(lines[:11]))
print(f"    ... {len(lines) - 13} more interaction terms ...")
print("\n".join(lines[-2:]))

california: 8 predictors -> 8 main effects + 28 interactions = 36 base learners

y ~ bbs(MedInc, df=4.0)
    + bbs(HouseAge, df=4.0)
    + bbs(AveRooms, df=4.0)
    + bbs(AveBedrms, df=4.0)
    + bbs(Population, df=4.0)
    + bbs(AveOccup, df=4.0)
    + bbs(Latitude, df=4.0)
    + bbs(Longitude, df=4.0)
    + bbs(MedInc, df=4.0, knots=10) %X% bbs(HouseAge, df=4.0, knots=10)
    + bbs(MedInc, df=4.0, knots=10) %X% bbs(AveRooms, df=4.0, knots=10)
    + bbs(MedInc, df=4.0, knots=10) %X% bbs(AveBedrms, df=4.0, knots=10)
    ... 23 more interaction terms ...
    + bbs(AveOccup, df=4.0, knots=10) %X% bbs(Longitude, df=4.0, knots=10)
    + bbs(Latitude, df=4.0, knots=10) %X% bbs(Longitude, df=4.0, knots=10)


In [6]:
mix = []
for name in ("california", "cpu_act", "credit_g"):
    _, Xd, _ = bench.encoded_datasets()[name]
    lv = levels_of(Xd)
    pr = [(a, b) for a in range(Xd.shape[1]) for b in range(a + 1, Xd.shape[1])]
    kinds = [pair_term(a, b, lv, 4.0, 10) for a, b in pr]
    mix.append({
        "dataset": name, "predictors": Xd.shape[1],
        "continuous": int((lv > 2).sum()), "indicator": int((lv <= 2).sum()),
        "pairs": len(pr),
        "tensor %X%": sum("%X%" in t for t in kinds),
        "varying by=": sum(t.startswith("bbs") and "%X%" not in t for t in kinds),
        "product bols": sum(t.startswith("bols") for t in kinds)})
display(annotated(
    pl.DataFrame(mix), stub="dataset", stubhead="Data set",
    title="Pair terms offered on three data sets",
    subtitle="How each pair of predictors enters the all-pairs formula",
    labels={"tensor %X%": "tensor", "varying by=": "varying by=", "product bols": "product"},
    notes=["tensor: two continuous predictors, bbs %X% bbs. varying by=: a continuous and "
           "an indicator predictor, a smooth of the first multiplied by the second. "
           "product: two indicators, a linear term in their product."]))

GT(_tbl_data=shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────┬────────────┬─────────┬────────────────┐
│ dataset    ┆ predictors ┆ continuous ┆ indicator ┆ pairs ┆ tensor %X% ┆ varying ┆ product bols   │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---   ┆ ---        ┆ by=     ┆ ---            │
│ str        ┆ i64        ┆ i64        ┆ i64       ┆ i64   ┆ i64        ┆ ---     ┆ i64            │
│            ┆            ┆            ┆           ┆       ┆            ┆ i64     ┆                │
╞════════════╪════════════╪════════════╪═══════════╪═══════╪════════════╪═════════╪════════════════╡
│ california ┆ 8          ┆ 8          ┆ 0         ┆ 28    ┆ 28         ┆ 0       ┆ 0              │
│ cpu_act    ┆ 21         ┆ 21         ┆ 0         ┆ 210   ┆ 210        ┆ 0       ┆ 0              │
│ credit_g   ┆ 59         ┆ 6          ┆ 53        ┆ 1711  ┆ 15         ┆ 318     ┆ 1378           │
└────────────┴────────────┴────────────┴───────────┴───────┴────────────┴─────────┴────────────────┘, _body=<great_tables._gt_data.Body object at 0x79faba7b4b60>, _boxhead=Boxhead([ColInfo(var='dataset', type=<ColInfoTypeEnum.stub: 2>, column_label='dataset', column_align='left', column_width=None, stub_level=0), ColInfo(var='predictors', type=<ColInfoTypeEnum.default: 1>, column_label='predictors', column_align='right', column_width=None, stub_level=None), ColInfo(var='continuous', type=<ColInfoTypeEnum.default: 1>, column_label='continuous', column_align='right', column_width=None, stub_level=None), ColInfo(var='indicator', type=<ColInfoTypeEnum.default: 1>, column_label='indicator', column_align='right', column_width=None, stub_level=None), ColInfo(var='pairs', type=<ColInfoTypeEnum.default: 1>, column_label='pairs', column_align='right', column_width=None, stub_level=None), ColInfo(var='tensor %X%', type=<ColInfoTypeEnum.default: 1>, column_label='tensor', column_align='right', column_width=None, stub_level=None), ColInfo(var='varying by=', type=<ColInfoTypeEnum.default: 1>, column_label='varying by=', column_align='right', column_width=None, stub_level=None), ColInfo(var='product bols', type=<ColInfoTypeEnum.default: 1>, column_label='product', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79fac08108c0>, _spanners=Spanners([]), _heading=Heading(title='Pair terms offered on three data sets', subtitle='How each pair of predictors enters the all-pairs formula', preheader=None), _stubhead='Data set', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79fabb6d0200>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79fabb457dd0>, _source_notes=['tensor: two continuous predictors, bbs %X% bbs. varying by=: a continuous and an indicator predictor, a smooth of the first multiplied by the second. product: two indicators, a linear term in their product.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x79fad0b79850>, _formats=[], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x79fabb74bd70>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helve

credit_g is where this matters: 53 of its 59 columns are one-hot indicators, so
only 15 of its 1711 pairs are genuine two-dimensional smooths.

The full sweep runs from `benches/bench_interaction_search.py`, which takes a
few hours; the results loaded above are its output. The fit below is one split
of one dataset, run here, so the machinery is visible. It is one
configuration on one split, so it does not reproduce the swept figure exactly —
that is an average over three splits with the configuration chosen by
cross-validation — but the two should be close.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from bench_interaction_search import SearchGAMBoost

task, Xc, yc = bench.encoded_datasets()["california"]
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.25,
                                      random_state=bench.SEED)
live = SearchGAMBoost(task, mode="all_pairs", mstop=8000, nu=0.1,
                      marginal_df=4.0, knots=10).fit(Xtr, ytr)
print(f"one split, one configuration, fitted here: "
      f"R2 = {r2_score(yte, live.predict(Xte)):.4f}")
print(f"the swept result for california, averaged over three splits and with "
      f"the configuration chosen by cross-validation: "
      f"{by_dataset.filter((pl.col('dataset') == 'california') & (pl.col('library') == 'mboostrs (all pairs)'))['score'][0]:.4f}")

one split, one configuration, fitted here: R2 = 0.7802
the swept result for california, averaged over three splits and with the configuration chosen by cross-validation: 0.7587


## Held-out score

Regression is scored by $R^2$, classification by ROC AUC. Predictions are held
to the range of the training responses for every model in the comparison, not
only mboostrs: a gradient-boosted tree's prediction is a sum of leaf increments
and can leave that range too, though a spline surface leaves it much further.

In [8]:
display(annotated(
    by_dataset.pivot(on="dataset", index="library", values="score")
    .with_columns(pl.col("library").replace_strict({k: i for i, k in enumerate(ORDER)}).alias("o"))
    .sort("o").drop("o").select("library", *DATASETS),
    stub="library", stubhead="Library",
    title="Held-out score with and without interaction terms",
    subtitle="Mean over 3 random 75/25 splits; predictions held to the training range",
    spanners={"R² ↑": DATASETS[:3], "AUC ↑": DATASETS[3:]},
    better={d: "max" for d in DATASETS}, decimals=4,
    notes=[SCORE_NOTE + " Bold: best on the data set."]))

GT(_tbl_data=shape: (8, 7)
┌──────────────────────┬────────────┬──────────┬──────────┬───────────────┬──────────┬─────────────┐
│ library              ┆ california ┆ cpu_act  ┆ diabetes ┆ breast_cancer ┆ credit_g ┆ electricity │
│ ---                  ┆ ---        ┆ ---      ┆ ---      ┆ ---           ┆ ---      ┆ ---         │
│ str                  ┆ f64        ┆ f64      ┆ f64      ┆ f64           ┆ f64      ┆ f64         │
╞══════════════════════╪════════════╪══════════╪══════════╪═══════════════╪══════════╪═════════════╡
│ mboostrs (all pairs) ┆ 0.75872    ┆ 0.985061 ┆ 0.486963 ┆ 0.996785      ┆ 0.771505 ┆ 0.873683    │
│ mboostrs (screened)  ┆ 0.749461   ┆ 0.983452 ┆ 0.48859  ┆ 0.989658      ┆ 0.780978 ┆ 0.867203    │
│ mboostrs GAMBoost    ┆ 0.718742   ┆ 0.980789 ┆ 0.499496 ┆ 0.995318      ┆ 0.789257 ┆ 0.85954     │
│ EBM                  ┆ 0.80842    ┆ 0.984704 ┆ 0.484613 ┆ 0.998463      ┆ 0.795479 ┆ 0.908458    │
│ XGBoost              ┆ 0.813615   ┆ 0.985283 ┆ 0.411659 ┆ 0.995388      ┆ 0.788978 ┆ 0.917252    │
│ LightGBM             ┆ 0.814093   ┆ 0.982822 ┆ 0.417331 ┆ 0.997973      ┆ 0.786794 ┆ 0.918378    │
│ CatBoost             ┆ 0.827827   ┆ 0.984909 ┆ 0.447118 ┆ 0.996995      ┆ 0.793194 ┆ 0.918455    │
│ MLP                  ┆ 0.732678   ┆ 0.97563  ┆ 0.452435 ┆ 0.991754      ┆ 0.767213 ┆ 0.853349    │
└──────────────────────┴────────────┴──────────┴──────────┴───────────────┴──────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x79fabb34ad50>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None, stub_level=0), ColInfo(var='california', type=<ColInfoTypeEnum.default: 1>, column_label='california', column_align='right', column_width=None, stub_level=None), ColInfo(var='cpu_act', type=<ColInfoTypeEnum.default: 1>, column_label='cpu_act', column_align='right', column_width=None, stub_level=None), ColInfo(var='diabetes', type=<ColInfoTypeEnum.default: 1>, column_label='diabetes', column_align='right', column_width=None, stub_level=None), ColInfo(var='breast_cancer', type=<ColInfoTypeEnum.default: 1>, column_label='breast_cancer', column_align='right', column_width=None, stub_level=None), ColInfo(var='credit_g', type=<ColInfoTypeEnum.default: 1>, column_label='credit_g', column_align='right', column_width=None, stub_level=None), ColInfo(var='electricity', type=<ColInfoTypeEnum.default: 1>, column_label='electricity', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79faba5b10d0>, _spanners=Spanners([SpannerInfo(spanner_id='R² ↑', spanner_level=0, spanner_label='R² ↑', spanner_units=None, spanner_pattern=None, vars=['california', 'cpu_act', 'diabetes'], built=None), SpannerInfo(spanner_id='AUC ↑', spanner_level=0, spanner_label='AUC ↑', spanner_units=None, spanner_pattern=None, vars=['breast_cancer', 'credit_g', 'electricity'], built=None)]), _heading=Heading(title='Held-out score with and without interaction terms', subtitle='Mean over 3 random 75/25 splits; predictions held to the training range', preheader=None), _stubhead='Library', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79faba475f70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79faba475ca0>, _source_notes=['Scores are the mean over three random 75/25 splits shared by every model: test R² for california, cpu_act and diabetes, test ROC AUC for breast_cancer, credit_g and electricity; both are higher-is-better with 1 the maximum. Bold: best on the data set.'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA476240>, mask=None), grpname=None, colname='california', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA476240>, mask=None), grpname=None, colname='california', rownum=1, c

In [9]:
display(annotated(
    by_dataset.with_columns((-pl.col("score")).rank("average").over("dataset").alias("rank"))
    .group_by("library").agg(pl.col("rank").mean().alias("mean rank"),
                             pl.col("score").mean().alias("mean score"))
    .sort("mean rank"),
    stub="library", stubhead="Library",
    title="Mean rank and mean score over the six data sets",
    subtitle="Rank 1 is the best library on a data set",
    labels={"mean rank": "mean rank ↓", "mean score": "mean score ↑"},
    better={"mean rank": "min", "mean score": "max"}, formats={"mean rank": 2},
    decimals=4, notes=["Bold: best in the column."]))

GT(_tbl_data=shape: (8, 3)
┌──────────────────────┬───────────┬────────────┐
│ library              ┆ mean rank ┆ mean score │
│ ---                  ┆ ---       ┆ ---        │
│ str                  ┆ f64       ┆ f64        │
╞══════════════════════╪═══════════╪════════════╡
│ CatBoost             ┆ 2.666667  ┆ 0.828083   │
│ EBM                  ┆ 3.0       ┆ 0.830023   │
│ LightGBM             ┆ 4.0       ┆ 0.819565   │
│ XGBoost              ┆ 4.0       ┆ 0.818696   │
│ mboostrs (all pairs) ┆ 4.333333  ┆ 0.81212    │
│ mboostrs GAMBoost    ┆ 5.333333  ┆ 0.80719    │
│ mboostrs (screened)  ┆ 5.5       ┆ 0.80989    │
│ MLP                  ┆ 7.166667  ┆ 0.79551    │
└──────────────────────┴───────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x79fb09b77050>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None, stub_level=0), ColInfo(var='mean rank', type=<ColInfoTypeEnum.default: 1>, column_label='mean rank ↓', column_align='right', column_width=None, stub_level=None), ColInfo(var='mean score', type=<ColInfoTypeEnum.default: 1>, column_label='mean score ↑', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79fabb902630>, _spanners=Spanners([]), _heading=Heading(title='Mean rank and mean score over the six data sets', subtitle='Rank 1 is the best library on a data set', preheader=None), _stubhead='Library', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79faba475f70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79faba477bf0>, _source_notes=['Bold: best in the column.'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA475820>, mask=None), grpname=None, colname='mean rank', rownum=4, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA475820>, mask=None), grpname=None, colname='mean rank', rownum=5, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA475820>, mask=None), grpname=None, colname='mean rank', rownum=6, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA475820>, mask=None), grpname=None, colname='mean score', rownum=4, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA475820>, mask=None), grpname=None, colname='mean score', rownum=5, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FABA475820>, mask=None), grpname=None, colname='mean score', rownum=6, colnum=None, styles=[CellStyleFill(color='#e8f0fb')])], _locale=<great_tables._gt_data.Locale object at 0x79faba476ba0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x79faba477080>, <great_tables._gt_data.FormatInfo object at 0x79faba475d30>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x79faba475fd0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, catego

Allowing interactions moves mboostrs from a mean rank of 5.33 to 4.33. It passes
the MLP and sits behind CatBoost, EBM, LightGBM and XGBoost. On cpu_act it
comes within 0.0002 of the best score in the field, and on diabetes it beats
every non-mboostrs model — though there the additive fit beats it in turn.

The screen's score is within 0.01 of the exhaustive search on all six datasets for
a fraction of the work. Its mean rank, 5.50, is behind the exhaustive search (4.33)
and also behind the additive fit (5.33): on three datasets it improves the score of
the additive fit (california, cpu_act, electricity) and on three it lowers it
(breast_cancer, credit_g, diabetes).

In [10]:
chart = alt.Chart(by_dataset).mark_bar().encode(
    y=alt.Y("library:N", sort=ORDER, title=None),
    x=alt.X("score:Q", title="held-out score", scale=alt.Scale(zero=False)),
    color=alt.Color("library:N", legend=None, scale=alt.Scale(domain=ORDER,
        range=["#e15759", "#ff9d9a", "#4e79a7", "#59a14f", "#f28e2b",
               "#edc948", "#b07aa1", "#76b7b2"]))
).properties(width=180, height=150).facet(
    facet=alt.Facet("dataset:N", title=None), columns=3, spacing=16
).resolve_scale(x="independent")
chart

alt.FacetChart(...)

## Results by dataset

In [11]:
wide = by_dataset.pivot(on="library", index="dataset", values="score")
display(annotated(wide.with_columns(
            (pl.col("mboostrs (all pairs)") - pl.col("mboostrs GAMBoost"))
                .alias("interactions gained"),
            pl.max_horizontal("EBM", "XGBoost", "LightGBM", "CatBoost", "MLP")
                .alias("best of the others"))
        .with_columns((pl.col("best of the others")
                       - pl.col("mboostrs (all pairs)")).alias("still behind"))
        .select("dataset", "mboostrs GAMBoost", "mboostrs (all pairs)",
                "interactions gained", "best of the others", "still behind")
        .sort("interactions gained", descending=True),
    stub="dataset", stubhead="Data set", decimals=4,
    title="What the interaction terms gain, and what remains",
    subtitle="Mean held-out score over 3 splits",
    labels={"mboostrs GAMBoost": "additive", "mboostrs (all pairs)": "all pairs",
            "interactions gained": "gained", "best of the others": "best other",
            "still behind": "still behind"},
    notes=[SCORE_NOTE, "gained: all pairs minus additive. best other: the best of EBM, "
           "XGBoost, LightGBM, CatBoost and the MLP. still behind: best other minus all "
           "pairs (negative when mboostrs leads)."]))

GT(_tbl_data=shape: (6, 6)
┌───────────────┬──────────┬───────────────┬──────────────┬────────────────────┬──────────────┐
│ dataset       ┆ mboostrs ┆ mboostrs (all ┆ interactions ┆ best of the others ┆ still behind │
│ ---           ┆ GAMBoost ┆ pairs)        ┆ gained       ┆ ---                ┆ ---          │
│ str           ┆ ---      ┆ ---           ┆ ---          ┆ f64                ┆ f64          │
│               ┆ f64      ┆ f64           ┆ f64          ┆                    ┆              │
╞═══════════════╪══════════╪═══════════════╪══════════════╪════════════════════╪══════════════╡
│ california    ┆ 0.718742 ┆ 0.75872       ┆ 0.039978     ┆ 0.827827           ┆ 0.069108     │
│ electricity   ┆ 0.85954  ┆ 0.873683      ┆ 0.014143     ┆ 0.918455           ┆ 0.044772     │
│ cpu_act       ┆ 0.980789 ┆ 0.985061      ┆ 0.004272     ┆ 0.985283           ┆ 0.000222     │
│ breast_cancer ┆ 0.995318 ┆ 0.996785      ┆ 0.001468     ┆ 0.998463           ┆ 0.001677     │
│ diabetes      ┆ 0.499496 ┆ 0.486963      ┆ -0.012533    ┆ 0.484613           ┆ -0.00235     │
│ credit_g      ┆ 0.789257 ┆ 0.771505      ┆ -0.017752    ┆ 0.795479           ┆ 0.023975     │
└───────────────┴──────────┴───────────────┴──────────────┴────────────────────┴──────────────┘, _body=<great_tables._gt_data.Body object at 0x79fb09517da0>, _boxhead=Boxhead([ColInfo(var='dataset', type=<ColInfoTypeEnum.stub: 2>, column_label='dataset', column_align='left', column_width=None, stub_level=0), ColInfo(var='mboostrs GAMBoost', type=<ColInfoTypeEnum.default: 1>, column_label='additive', column_align='right', column_width=None, stub_level=None), ColInfo(var='mboostrs (all pairs)', type=<ColInfoTypeEnum.default: 1>, column_label='all pairs', column_align='right', column_width=None, stub_level=None), ColInfo(var='interactions gained', type=<ColInfoTypeEnum.default: 1>, column_label='gained', column_align='right', column_width=None, stub_level=None), ColInfo(var='best of the others', type=<ColInfoTypeEnum.default: 1>, column_label='best other', column_align='right', column_width=None, stub_level=None), ColInfo(var='still behind', type=<ColInfoTypeEnum.default: 1>, column_label='still behind', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79faba43e960>, _spanners=Spanners([]), _heading=Heading(title='What the interaction terms gain, and what remains', subtitle='Mean held-out score over 3 splits', preheader=None), _stubhead='Data set', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79faba43f200>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79faba477b90>, _source_notes=['Scores are the mean over three random 75/25 splits shared by every model: test R² for california, cpu_act and diabetes, test ROC AUC for breast_cancer, credit_g and electricity; both are higher-is-better with 1 the maximum.', 'gained: all pairs minus additive. best other: the best of EBM, XGBoost, LightGBM, CatBoost and the MLP. still behind: best other minus all pairs (negative when mboostrs leads).'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x79faba474b30>, _formats=[<great_tables._gt_data.FormatInfo object at 0x79faba43ed20>, <great_tables._gt_data.FormatInfo object at 0x79faba4748c0>, <great_tables._gt_data.FormatInfo object at 0x79faba4750d0>, <great_tables._gt_data.FormatInfo object at 0x79faba4740e0>, <great_tables._gt_data.FormatInfo object at 0x79faba4778c0>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x79faba477350>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table',

california gains the most and remains the largest shortfall: interactions are
worth 0.040 and 0.069 of the gap survives them. Geography enters through
latitude interacting with longitude, and a smooth tensor surface represents
California's sharp price boundaries less well than axis-aligned splits do.
The surviving 0.069 is a property of the pairs-only configuration, not of the
library's reach. The next section fits the library's own search on these same
splits: the default route, which never offers the residual tree (a tree fitted to what the additive model leaves unexplained, so that it can capture interactions), and the same
route with the residual tree forced on.

Interactions lose ground on two datasets, both small. On diabetes they cost
0.013 with 331 training rows, and on credit_g 0.018, making the exhaustive search the
second weakest of the eight, above only the MLP, where the additive fit was mid-field. Offering 1,653 candidate pairs to a
selection step running on 750 training rows finds structure that does not reach
the held-out quarter, and nothing in the fitted model marks the difference. The
screen, which offers twenty, loses 0.008.


### The residual tree on california

The configurations above offer the boosting only smooth terms and their
pairwise products. The library's search (`autospec`, budget 200) is a
different route through the same library: it measures shapes, pairs and
a coordinate surface on training folds, and it can add one more term —
a depth-5 regression tree over all the columns (`residual_tree`), which
splits on latitude and longitude directly instead of smoothing over
them. The fits below use the same capped rows and
the same three 75/25 splits as the recorded table, with predictions
held to the training response range, as everywhere on this page. Two
configurations: the default route, exactly as the library ships it,
and the same route with `residual_tree=True`, which offers the tree on
every split and lets the held-out comparison decide whether it stays.


In [ ]:
import time

import mboostrs as mb
from sklearn.datasets import fetch_california_housing

# bench.datasets() applies the benchmark's own 5000-row cap, and each
# split below is the recorded one: train_test_split with
# random_state=SEED + repeat.
_task, X_cal, y_cal = bench.datasets()["california"]
FEATURES = list(fetch_california_housing().feature_names)
california_frame = pl.DataFrame(
    {name: X_cal[:, j] for j, name in enumerate(FEATURES)}
).with_columns(pl.Series("MedHouseVal", y_cal))




In [ ]:
def autospec_route(train, test, seed, **kwargs):
    rec = mb.autospec(train, "MedHouseVal", budget=200, seed=seed, **kwargs)
    fit = rec.fit(train, na_action="impute")
    pred = fit.predict(newdata=test, type="response",
                       unseen_levels="zero_effect")
    if isinstance(pred, dict):
        # A distributional (LSS) fit predicts its parameters; the score
        # is on the mean, the "mu" component.
        pred = pred["mu"]
    return rec, np.asarray(pred, dtype=float)




In [ ]:
def run_arm(label, **kwargs):
    rows = []
    # Use the recorded splits and training-range clipping for each route.
    for repeat in range(3):
        idx = np.arange(len(y_cal))
        Xtr, Xte, ytr, yte, itr, ite = train_test_split(
            X_cal, y_cal, idx, test_size=0.25,
            random_state=bench.SEED + repeat)
        train = california_frame[itr]
        test = california_frame[ite].drop("MedHouseVal")
        t0 = time.perf_counter()
        rec, pred = autospec_route(train, test, bench.SEED + repeat, **kwargs)
        seconds = time.perf_counter() - t0
        pred = np.clip(pred, float(np.min(ytr)), float(np.max(ytr)))
        tree_ev = rec.evidence.get("residual_tree") or {}
        rows.append({
            "repeat": repeat, "configuration": label,
            "score": float(r2_score(yte, pred)), "seconds": seconds,
            "family": type(rec.family).__name__,
            "offer": rec.evidence.get("residual_tree_offer"),
            "tree_adopted": tree_ev.get("adopted"),
            "held_out_gain": tree_ev.get("held_out_gain"),
        })
        print(f"repeat {repeat} {label}: R2 {rows[-1]['score']:.4f}, "
              f"family={rows[-1]['family']}, offer={rows[-1]['offer']!r}, "
              f"tree adopted={rows[-1]['tree_adopted']}, "
              f"held-out gain={rows[-1]['held_out_gain']}")
    return pl.DataFrame(rows)




In [12]:
rt_default = run_arm("autospec default route")
rt_default


repeat 0 autospec default route: R2 0.7949, family=Gaussian, offer=None, tree adopted=None, held-out gain=None


repeat 1 autospec default route: R2 0.7243, family=GaussianLSS, offer=None, tree adopted=None, held-out gain=None


repeat 2 autospec default route: R2 0.7574, family=Gaussian, offer=None, tree adopted=None, held-out gain=None


repeat,configuration,score,seconds,family,offer,tree_adopted,held_out_gain
i64,str,f64,f64,str,null,null,null
0,"""autospec default route""",0.794886,395.795473,"""Gaussian""",null,null,null
1,"""autospec default route""",0.72427,114.96174,"""GaussianLSS""",null,null,null
2,"""autospec default route""",0.757436,214.942273,"""Gaussian""",null,null,null


In [13]:
rt_forced = run_arm("autospec + residual tree", residual_tree=True)
rt = pl.concat([rt_default, rt_forced], how="vertical_relaxed")
rt


repeat 0 autospec + residual tree: R2 0.8164, family=Gaussian, offer=None, tree adopted=True, held-out gain=0.041185


repeat 1 autospec + residual tree: R2 0.7243, family=GaussianLSS, offer=None, tree adopted=False, held-out gain=None


repeat 2 autospec + residual tree: R2 0.7451, family=GaussianLSS, offer=None, tree adopted=True, held-out gain=None


repeat,configuration,score,seconds,family,offer,tree_adopted,held_out_gain
i64,str,f64,f64,str,null,bool,f64
0,"""autospec default route""",0.794886,395.795473,"""Gaussian""",null,null,null
1,"""autospec default route""",0.72427,114.96174,"""GaussianLSS""",null,null,null
2,"""autospec default route""",0.757436,214.942273,"""Gaussian""",null,null,null
0,"""autospec + residual tree""",0.816416,656.977096,"""Gaussian""",null,true,0.041185
1,"""autospec + residual tree""",0.72427,161.892902,"""GaussianLSS""",null,false,null
2,"""autospec + residual tree""",0.745123,335.138057,"""GaussianLSS""",null,true,null


In [14]:
rt_summary = rt.group_by("configuration").agg(
    pl.col("score").mean(), pl.col("seconds").mean())
cal_recorded = (wide.filter(pl.col("dataset") == "california")
                .with_columns(pl.max_horizontal(
                    "EBM", "XGBoost", "LightGBM", "CatBoost", "MLP")
                    .alias("best of the others")))
rt_table = pl.DataFrame({
    "configuration": ["additive (recorded)", "all pairs (recorded)",
                      "autospec default route", "autospec + residual tree",
                      "best other (recorded)"],
    "score": [cal_recorded["mboostrs GAMBoost"][0],
              cal_recorded["mboostrs (all pairs)"][0],
              rt_summary.filter(pl.col("configuration") == "autospec default route")["score"][0],
              rt_summary.filter(pl.col("configuration") == "autospec + residual tree")["score"][0],
              cal_recorded["best of the others"][0]],
}).with_columns(
    (pl.col("score").max() - pl.col("score")).alias("behind best"))
display(annotated(
    rt_table, stub="configuration", stubhead="Configuration", decimals=4,
    title="California: the search route and the residual tree",
    subtitle="Mean held-out R\u00b2 over the same 3 splits; recorded rows are the swept runs above",
    notes=["The two autospec rows are fitted in this notebook on the current "
           "tree (budget 200, predictions held to the training range). "
           "behind best: the best score in this table minus the row's score."]))
rt_summary


GT(_tbl_data=shape: (5, 3)
┌──────────────────────────┬──────────┬─────────────┐
│ configuration            ┆ score    ┆ behind best │
│ ---                      ┆ ---      ┆ ---         │
│ str                      ┆ f64      ┆ f64         │
╞══════════════════════════╪══════════╪═════════════╡
│ additive (recorded)      ┆ 0.718742 ┆ 0.109085    │
│ all pairs (recorded)     ┆ 0.75872  ┆ 0.069108    │
│ autospec default route   ┆ 0.758864 ┆ 0.068963    │
│ autospec + residual tree ┆ 0.761936 ┆ 0.065891    │
│ best other (recorded)    ┆ 0.827827 ┆ 0.0         │
└──────────────────────────┴──────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x79fac03a8110>, _boxhead=Boxhead([ColInfo(var='configuration', type=<ColInfoTypeEnum.stub: 2>, column_label='configuration', column_align='left', column_width=None, stub_level=0), ColInfo(var='score', type=<ColInfoTypeEnum.default: 1>, column_label='score', column_align='right', column_width=None, stub_level=None), ColInfo(var='behind best', type=<ColInfoTypeEnum.default: 1>, column_label='behind best', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79fae80b2180>, _spanners=Spanners([]), _heading=Heading(title='California: the search route and the residual tree', subtitle='Mean held-out R² over the same 3 splits; recorded rows are the swept runs above', preheader=None), _stubhead='Configuration', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79fb090d0c50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79fae38804a0>, _source_notes=["The two autospec rows are fitted in this notebook on the current tree (budget 200, predictions held to the training range). behind best: the best score in this table minus the row's score."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x79fabb34ad50>, _formats=[<great_tables._gt_data.FormatInfo object at 0x79fab02c2a80>, <great_tables._gt_data.FormatInfo object at 0x79fab02c19a0>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x79fab02c0e60>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_b

configuration,score,seconds
str,f64,f64
"""autospec + residual tree""",0.761936,384.669351
"""autospec default route""",0.758864,241.899829


On these splits the default search route scores 0.7589 — the same as the
all-pairs configuration (0.7587) and 0.040 above the additive fit
(0.7187). Forcing the residual tree on adds 0.003 on average (0.7619):
the tree is adopted on two of the three splits, gaining 0.022 on
repeat 0 and losing 0.012 on repeat 2, and on repeat 1 it is not
adopted and the score is unchanged. The gap to the best of the other
libraries (0.8278, recorded) is 0.069 for the default route and 0.066
with the tree forced on. On these splits the residual-tree gain is small
and uncertain. Scores from the other benchmarks in the book are not directly comparable with these, because those use different data (OpenML, a dollar target,
80/20 splits). In this comparison, the 0.069 that survives
the pair terms describes the pairs-only configuration, not the library's
full capabilities. On repeats 1
and 2 both routes take the distributional family (GaussianLSS, a location-scale model in which the mean and the standard deviation each have their own predictor); the
scores are for its mean. The forced route costs about 385 seconds
per fit against the default route's 242.


## Computing time

In [15]:
display(annotated(
    runs.group_by("library").agg(pl.col("total_seconds").mean().alias("seconds"))
    .sort("seconds"),
    stub="library", stubhead="Library",
    title="Seconds to tune and refit", subtitle="Mean over data sets and splits",
    labels={"seconds": "s ↓"}, better={"seconds": "min"}, formats={"seconds": 1},
    notes=["The mboostrs interaction variants and the other libraries come from separate "
           "runs; compare the variants with each other in the next table. Bold: fastest."]))
# Costs from the same interaction-search run share its fit boundary.
display(annotated(
    search.group_by("variant").agg(pl.col("total_seconds").mean().alias("seconds"))
    .sort("seconds"),
    stub="variant", stubhead="mboostrs variant",
    title="Seconds per mboostrs variant, same run",
    subtitle="Mean over data sets and splits, tuning and refit",
    labels={"seconds": "s ↓"}, better={"seconds": "min"}, formats={"seconds": 1},
    notes=["additive: main effects only. screened: pairs chosen by a residual screen. "
           "all_pairs: every pair offered to the boosting."]))


GT(_tbl_data=shape: (8, 2)
┌──────────────────────┬────────────┐
│ library              ┆ seconds    │
│ ---                  ┆ ---        │
│ str                  ┆ f64        │
╞══════════════════════╪════════════╡
│ mboostrs GAMBoost    ┆ 0.947222   │
│ MLP                  ┆ 6.215      │
│ CatBoost             ┆ 14.863889  │
│ mboostrs (screened)  ┆ 18.477778  │
│ XGBoost              ┆ 18.665556  │
│ LightGBM             ┆ 36.805556  │
│ EBM                  ┆ 106.913889 │
│ mboostrs (all pairs) ┆ 380.592222 │
└──────────────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x79fabb34ad50>, _boxhead=Boxhead([ColInfo(var='library', type=<ColInfoTypeEnum.stub: 2>, column_label='library', column_align='left', column_width=None, stub_level=0), ColInfo(var='seconds', type=<ColInfoTypeEnum.default: 1>, column_label='s ↓', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79fab02c0080>, _spanners=Spanners([]), _heading=Heading(title='Seconds to tune and refit', subtitle='Mean over data sets and splits', preheader=None), _stubhead='Library', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79fac0b49c10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79fae38804a0>, _source_notes=['The mboostrs interaction variants and the other libraries come from separate runs; compare the variants with each other in the next table. Bold: fastest.'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FAB02C2F60>, mask=None), grpname=None, colname='seconds', rownum=0, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FAB02C2F60>, mask=None), grpname=None, colname='seconds', rownum=3, colnum=None, styles=[CellStyleFill(color='#e8f0fb')]), StyleInfo(locname=LocBody(columns=None, rows=<Expr ['col("library").str.starts_with…'] at 0x79FAB02C2F60>, mask=None), grpname=None, colname='seconds', rownum=7, colnum=None, styles=[CellStyleFill(color='#e8f0fb')])], _locale=<great_tables._gt_data.Locale object at 0x79fab02c1220>, _formats=[<great_tables._gt_data.FormatInfo object at 0x79fab02c2750>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x79fab02c1bb0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(sc

GT(_tbl_data=shape: (3, 2)
┌───────────┬────────────┐
│ variant   ┆ seconds    │
│ ---       ┆ ---        │
│ str       ┆ f64        │
╞═══════════╪════════════╡
│ additive  ┆ 3.413889   │
│ screened  ┆ 18.477778  │
│ all_pairs ┆ 380.592222 │
└───────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x79fabb34ad50>, _boxhead=Boxhead([ColInfo(var='variant', type=<ColInfoTypeEnum.stub: 2>, column_label='variant', column_align='left', column_width=None, stub_level=0), ColInfo(var='seconds', type=<ColInfoTypeEnum.default: 1>, column_label='s ↓', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x79fae38804a0>, _spanners=Spanners([]), _heading=Heading(title='Seconds per mboostrs variant, same run', subtitle='Mean over data sets and splits, tuning and refit', preheader=None), _stubhead='mboostrs variant', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x79fab02c1bb0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x79fab02c18b0>, _source_notes=['additive: main effects only. screened: pairs chosen by a residual screen. all_pairs: every pair offered to the boosting.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x79fab02c1af0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x79fab02c1a30>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x79fab02c2cc0>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_

The same-run timing table gives 3.4 seconds for the additive fit, 18.5
seconds for the screened fit and 380.6 seconds when every pair is offered.
The exhaustive fit therefore takes about 111 times as long as its additive
control; screening cuts that cost by a factor of about 21 while giving up
1.17 mean-rank places. The first timing table includes models from the
companion benchmark, whose fits were measured separately, so their ratios
are less controlled.

The cost is the candidate count. A p-column dataset has p(p−1)/2 pairs, every
one is a base learner, and component-wise boosting scores every learner at every
iteration. A tree can instead consider different split variables along one fitted tree
without preparing a separate learner for every pair.

## Summary

Offering pairwise terms improves the mboostrs mean rank and held-out score on the datasets where interactions matter. The exhaustive pair search is much slower than the additive fit; screening lowers the cost and retains most of the score gain. The comparison is about these candidate spaces and tuning grids, with three common splits per dataset.

## Limitations

Six datasets capped at 5000 rows, three splits each, twelve tuning
configurations per model from hand-written grids of matched generosity. That
separates 0.07 on california and does not separate 0.002 on breast_cancer; no
ordering here distinguishes models finishing within a few thousandths.

Only two-way interactions were offered to the pair configurations. The tree
ensembles fit three-way and higher, so part of the remaining gap is a
restriction that still stands for those configurations; the residual-tree
section above shows what the search reaches when it may add a tree term.

Accuracy is the only thing scored. mboostrs and EBM report a fitted effect per
term, and the selected mboostrs pairs can be read term by term; the tree
ensembles and the MLP do not provide the same term-level display. Held-out
accuracy alone does not quantify the value of that difference.